In [2]:
# Imports und small helper: prüfe, ob wichtige Pakete vorhanden sind und zeige eine kurze Nachricht,
# falls etwas fehlt, installiere es in deiner Virtualenv (zsh):
# pip install ipywidgets scikit-image matplotlib imageio
import pickle
import gzip
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, VBox
import os

print('Imports OK')

Imports OK


### Helper functions

In [3]:
def load_zipped_pickle(filename):
    with gzip.open(filename, 'rb') as f:
        loaded_object = pickle.load(f)
        return loaded_object

In [4]:
def save_zipped_pickle(obj, filename):
    with gzip.open(filename, 'wb') as f:
        pickle.dump(obj, f, 2)

In [6]:
# Pfad zu train.pkl anpassen falls nötig
pkl_path = '../data/raw/train.pkl'
if not os.path.exists(pkl_path):
    raise FileNotFoundError(f'{pkl_path} nicht gefunden im aktuellen Verzeichnis: {os.getcwd()}')

data = load_zipped_pickle(pkl_path)

print('Number of samples:', len(data))

# Inspect a few samples (up to 3)
for i, sample in enumerate(data[:3]):
    print(f'Sample {i} name:', sample.get('name'))
    vid = sample.get('video')
    lab = sample.get('label')
    box = sample.get('box')
    print('  video: dtype=', getattr(vid, 'dtype', None), ' shape=', getattr(vid, 'shape', None))
    print('  label: dtype=', getattr(lab, 'dtype', None), ' shape=', getattr(lab, 'shape', None))
    print('  box:   dtype=', getattr(box, 'dtype', None), ' shape=', getattr(box, 'shape', None))
    print('  frames:', sample.get('frames'))

Number of samples: 65
Sample 0 name: D47OR19ANJ
  video: dtype= uint8  shape= (112, 112, 334)
  label: dtype= bool  shape= (112, 112, 334)
  box:   dtype= bool  shape= (112, 112)
  frames: [15, 59, 143]
Sample 1 name: IMNKTJV3OI
  video: dtype= uint8  shape= (112, 112, 177)
  label: dtype= bool  shape= (112, 112, 177)
  box:   dtype= bool  shape= (112, 112)
  frames: [2, 47, 79]
Sample 2 name: YSCCEISFRH
  video: dtype= uint8  shape= (112, 112, 195)
  label: dtype= bool  shape= (112, 112, 195)
  box:   dtype= bool  shape= (112, 112)
  frames: [18, 83, 131]


In [9]:
# Interaktiver Viewer mit ipywidgets + matplotlib
# Funktion zum Anzeigen eines Frames mit Masken-Overlay und Konturen
def show_frame(sample_idx=0, t=0, alpha=0.5, downsample=1):
    sample = data[sample_idx]
    video = sample['video']  # video shape (H, W, T)
    label = sample.get('label')
    box = sample.get('box')

    # handle possible color channels
    img = video[:, :, t]

    mask = None
    if label is not None:
        mask = label[:, :, t].astype(bool)

    plt.figure(figsize=(6,6))
    plt.imshow(img, cmap='gray')

    # mask overlay (red) — uses per-pixel RGBA so alpha only affects mask area
    if mask is not None:
        h, w = mask.shape
        rgba = np.zeros((h, w, 4), dtype=np.float32)
        rgba[..., 0] = 1.0                      # red channel
        rgba[..., 3] = mask.astype(float) * alpha  # alpha channel only where mask True
        plt.imshow(rgba, interpolation='nearest')
        # draw contours for better visibility
        try:
            contours = measure.find_contours(mask.astype(np.uint8), 0.5)
            for contour in contours:
                plt.plot(contour[:, 1], contour[:, 0], linewidth=1.2, color='yellow')
        except Exception:
            # falls find_contours fehlschlägt, ignoriere
            pass

    # box overlay (green) — assume box is a 2D bool mask (H, W)
    if box is not None:
        b = np.asarray(box).astype(bool)
        # make overlay mask
        rgba_box = np.zeros((b.shape[0], b.shape[1], 4), dtype=np.float32)
        rgba_box[..., 1] = 1.0  # green channel
        rgba_box[..., 3] = b.astype(float) * alpha / 2
        plt.imshow(rgba_box, interpolation='nearest')

    plt.title(f"{sample.get('name', 'sample')} frame {t}. Dataset: {sample['dataset']}")
    plt.axis('off')
    plt.show()

# Widgets: sample index, frame index (aktualisiert sich beim Wechsel des Samples), alpha
n_samples = len(data)
sample_slider = IntSlider(min=0, max=n_samples-1, value=0, description='sample')
# initial max für frame von sample 0
initial_max_frame = data[0]['video'].shape[0] - 1
frame_slider = IntSlider(min=0, max=initial_max_frame, value=0, description='frame')
alpha_slider = FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='alpha')
downsample_slider = IntSlider(min=1, max=8, step=1, value=1, description='down')

def _on_sample_change(change):
    if change['name'] == 'value':
        s = change['new']
        maxf = data[s]['video'].shape[2] - 1
        frame_slider.max = maxf
        frame_slider.value = 0

sample_slider.observe(_on_sample_change, names='value')

out = widgets.interactive_output(show_frame, {'sample_idx': sample_slider, 't': frame_slider, 'alpha': alpha_slider, 'downsample': downsample_slider})
ui = VBox([sample_slider, frame_slider, alpha_slider, downsample_slider])
display(ui, out)


Output()